# Phase 5 — Huấn luyện NER PhoBERT (arm `large`)

- **Model**: `vinai/phobert-large`
- **Dataset**: `dathungdao2004/vhgr-ner-data` (private)
- **Task**: token classification, scheme **BIO 13 lớp** (6 nhãn × {B, I} + `O`)
- **Seed**: 42 (theo `AGENTS.md` §5.6)

Máy local không có GPU → toàn bộ huấn luyện chạy ở đây (Kaggle T4 ×2).

## Quy trình

1. Mount dataset → giải nén nhãn bạc + mã nguồn + weight model
2. Kiểm tra thư viện (kernel không có DNS → không `pip install` được)
3. Kiểm tra GPU + môi trường
4. Huấn luyện (`src.ner.train_ner`)
5. Suy luận toàn corpus (`src.ner.infer_ner`) → `ner_predictions.jsonl`
6. Kiểm tra offset (0 span sai) + đóng gói output


In [ ]:
# --- 1. Mount dataset ---------------------------------------------------------
import gzip
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

MODEL_NAME = "vinai/phobert-large"
ARM_NAME = "large"

WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
REPO = WORK / "repo"

# Tìm thư mục dataset đã mount: Kaggle đặt tên theo slug, có thể kèm hậu tố.
def find_dataset_dir() -> Path:
    candidates = sorted(
        path
        for path in INPUT_ROOT.iterdir()
        if path.is_dir() and "vhgr-ner-data" in path.name
    )
    if not candidates:
        # Fallback: dataset được giải nén ra thư mục con.
        candidates = sorted(
            path
            for path in INPUT_ROOT.rglob("ner_silver.jsonl*")
            if path.parent.is_dir()
        )
        if not candidates:
            raise FileNotFoundError(
                f"Không tìm thấy dataset trong {INPUT_ROOT}: {list(INPUT_ROOT.iterdir())}"
            )
        return candidates[0].parent
    return candidates[0]


DATASET_DIR = find_dataset_dir()
print("Dataset mount tại:", DATASET_DIR)
print("Nội dung:", sorted(p.name for p in DATASET_DIR.iterdir()))

if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True, exist_ok=True)

for item in DATASET_DIR.iterdir():
    if item.name.endswith(".zip") and item.is_file():
        with zipfile.ZipFile(item) as archive:
            archive.extractall(REPO)
    elif item.is_dir():
        shutil.copytree(item, REPO / item.name, dirs_exist_ok=True)
    else:
        shutil.copy2(item, REPO / item.name)

# Giải nén nhãn bạc
silver_gz = REPO / "ner_silver.jsonl.gz"
silver_plain = REPO / "ner_silver.jsonl"
if silver_gz.exists() and not silver_plain.exists():
    with gzip.open(silver_gz, "rb") as src, silver_plain.open("wb") as dst:
        shutil.copyfileobj(src, dst)

# Đặt đúng cấu trúc mà `src.common.config` kỳ vọng
(REPO / "data" / "interim").mkdir(parents=True, exist_ok=True)
(REPO / "data" / "splits").mkdir(parents=True, exist_ok=True)
if silver_plain.exists():
    shutil.move(str(silver_plain), str(REPO / "data" / "interim" / "ner_silver.jsonl"))
if (REPO / "splits.json").exists():
    shutil.move(str(REPO / "splits.json"), str(REPO / "data" / "splits" / "splits.json"))

print("Nội dung repo:", sorted(p.name for p in REPO.iterdir()))
print(
    "Số câu nhãn bạc:",
    sum(1 for _ in (REPO / "data" / "interim" / "ner_silver.jsonl").open(encoding="utf-8")),
)

# --- Model: dùng weight ĐÃ BUNDLE trong dataset -------------------------------
# Kernel Kaggle không có DNS → không tải được từ huggingface.co.
MODEL_DIR = REPO / "models_hf" / MODEL_NAME.split("/")[-1]
assert MODEL_DIR.exists(), f"Không thấy weight model tại {MODEL_DIR}"
MODEL_PATH = str(MODEL_DIR)
print("Model path:", MODEL_PATH)
print("Model files:", sorted(p.name for p in MODEL_DIR.iterdir()))


In [ ]:
# --- 2. Cài thư viện ------------------------------------------------------------
# Kernel không có DNS → `pip install` sẽ thất bại. Thư viện cần đã có sẵn
# trong ảnh Kaggle; chỉ kiểm tra, không cài.
import importlib

import torch
import transformers

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
for module in ["accelerate", "numpy", "sklearn", "sentencepiece", "tokenizers"]:
    try:
        imported = importlib.import_module(module)
        print(f"{module}: {getattr(imported, '__version__', '?')}")
    except Exception as exc:
        print(f"{module}: THIẾU ({exc})")


In [ ]:
# --- 3. Kiểm tra môi trường ----------------------------------------------------
import sys

sys.path.insert(0, "/kaggle/working/repo")

assert torch.cuda.is_available(), "Không có GPU — kernel phải chạy với accelerator GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Số GPU:", torch.cuda.device_count())

from src.ner.data_convert import (
    LABEL_LIST,
    NUM_LABELS,
    load_tokenizer,
)

print("Số lớp NER:", NUM_LABELS)
print("Danh sách lớp:", LABEL_LIST)

# Tokenizer: vocab phải khớp kích thước embedding của model (D-040)
tokenizer = load_tokenizer(MODEL_PATH)
print("Vocab tokenizer:", tokenizer.vocab_size)

import json
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_PATH)
print("vocab_size model:", config.vocab_size)
assert tokenizer.vocab_size <= int(config.vocab_size), "Vocab tokenizer > embedding model"


In [ ]:
# --- 4. Huấn luyện --------------------------------------------------------------
from src.ner.train_ner import main as train_main

train_main(
    [
        "--config", "/kaggle/working/repo/configs/default.yaml",
        "--model", MODEL_PATH,
        "--silver", "/kaggle/working/repo/data/interim/ner_silver.jsonl",
        "--splits", "/kaggle/working/repo/data/splits/splits.json",
        "--out", "/kaggle/working/models/ner",
        "--batch-size", "8",
        "--eval-batch-size", "16",
        "--grad-accum", "4",
        "--epochs", "2",
        "--max-train", "60000",
    ]
)


In [ ]:
# --- 6. Kiểm tra offset + đóng gói ------------------------------------------------
import json
import shutil
from pathlib import Path

PRED = Path("/kaggle/working/ner_predictions.jsonl")
VALID_LABELS = {"PERSON", "DYNASTY", "EVENT", "LOCATION", "TIME", "TITLE"}

n_sentences = 0
n_entities = 0
n_bad_offset = 0
n_bad_label = 0

if PRED.exists():
    with PRED.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            n_sentences += 1
            text = record["text"]
            for entity in record["entities"]:
                n_entities += 1
                if text[entity["start"] : entity["end"]] != entity["text"]:
                    n_bad_offset += 1
                if entity["label"] not in VALID_LABELS:
                    n_bad_label += 1

    print(f"Câu: {n_sentences}")
    print(f"Entity: {n_entities}")
    print(f"Span sai offset: {n_bad_offset}")
    print(f"Nhãn không hợp lệ: {n_bad_label}")

    assert n_bad_offset == 0, "Có span sai offset — Gate Phase 5 không đạt"
    assert n_bad_label == 0, "Có nhãn không hợp lệ"

# Đóng gói model + metrics + predictions thành 1 zip để tải về
shutil.make_archive("/kaggle/working/vhgr_ner_output", "zip", "/kaggle/working", "models")
if PRED.exists():
    shutil.make_archive(
        "/kaggle/working/vhgr_ner_predictions", "zip", "/kaggle/working", "ner_predictions.jsonl"
    )
print("Đã đóng gói output.")


In [ ]:
# --- 7. Tóm tắt -----------------------------------------------------------------
import json
from pathlib import Path

metrics = json.loads(Path("/kaggle/working/models/ner/metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=1))
